In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate requests fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import json
import re
import threading
import time

import requests
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import uvicorn

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
OPEN_LIBRARY_URL = "https://openlibrary.org/search.json"


def search_books(query, limit=30):
    fields = ",".join([
        "key",
        "title",
        "author_name",
        "first_publish_year",
        "subject",
        "language",
        "number_of_pages_median",
        "cover_i",
        "ratings_count",
        "edition_count"
    ])

    params = {
        "q": query,
        "fields": fields,
        "limit": limit
    }

    response = requests.get(
        OPEN_LIBRARY_URL,
        params=params,
        timeout=15
    )

    response.raise_for_status()

    return response.json()

In [ ]:
def normalize_book(book):
    title = book.get("title", "Unknown Title")

    authors = book.get("author_name", [])
    if isinstance(authors, list):
        author = ", ".join(authors[:3])
    else:
        author = str(authors)

    subjects = book.get("subject", [])
    if isinstance(subjects, list):
        subjects = subjects[:15]
    else:
        subjects = [str(subjects)]

    languages = book.get("language", [])
    if isinstance(languages, list):
        languages = languages[:10]
    else:
        languages = [str(languages)]

    return {
        "id": book.get("key"),
        "title": title,
        "author": author,
        "year": book.get("first_publish_year"),
        "subjects": subjects,
        "languages": languages,
        "pages": book.get("number_of_pages_median"),
        "cover_id": book.get("cover_i"),
        "ratings_count": book.get("ratings_count", 0),
        "edition_count": book.get("edition_count", 0)
    }

In [ ]:
def book_to_text(book):
    subjects = ", ".join(book.get("subjects", []))

    return (
        f"Title: {book.get('title', '')}. "
        f"Author: {book.get('author', '')}. "
        f"Genres and subjects: {subjects}. "
        f"Publication year: {book.get('year', '')}."
    )

In [ ]:
EMBEDDING_DIM = 384

faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)

print("FAISS index created.")

In [ ]:
def create_embeddings(texts):
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    return embeddings

In [ ]:
def build_book_index(books):
    texts = [book_to_text(book) for book in books]

    embeddings = create_embeddings(texts)

    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(embeddings)

    return index, embeddings

In [ ]:
def search_similar_books(query, books, index, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )

    scores, indices = index.search(
        query_embedding,
        min(top_k, len(books))
    )

    results = []

    for score, index_position in zip(scores[0], indices[0]):
        if index_position == -1:
            continue

        book = books[index_position].copy()
        book["similarity"] = float(score)

        results.append(book)

    return results

In [ ]:
#test
raw_results = search_books(
    "dark mystery detective",
    limit=40
)

books = [
    normalize_book(book)
    for book in raw_results.get("docs", [])
]

print("Books retrieved:", len(books))

In [ ]:
index, embeddings = build_book_index(books)

print("Books indexed:", index.ntotal)
print("Embedding dimension:", embeddings.shape[1])

In [ ]:
recommendations = search_similar_books(
    "I want a dark mystery with a detective and suspense.",
    books,
    index,
    top_k=10
)

for i, book in enumerate(recommendations, start=1):
    print(
        f"{i}. {book['title']} — {book['author']} "
        f"(similarity: {book['similarity']:.3f})"
    )

In [ ]:
def get_cover_url(book):
    cover_id = book.get("cover_id")

    if not cover_id:
        return None

    return f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg"

In [ ]:
def get_book_url(book):
    book_id = book.get("id")

    if not book_id:
        return None

    return f"https://openlibrary.org{book_id}"

In [ ]:
def find_reference_book(title):
    results = search_books(title, limit=10)

    docs = results.get("docs", [])

    if not docs:
        return None

    return normalize_book(docs[0])

In [ ]:
reference_book = find_reference_book("The Alchemist")

print(reference_book)

In [ ]:
#Generate recommendations based on a reference book
def recommend_similar_to_book(reference_book, limit=40, top_k=10):
    search_query = reference_book["title"]

    raw_results = search_books(
        search_query,
        limit=limit
    )

    candidates = [
        normalize_book(book)
        for book in raw_results.get("docs", [])
    ]

    candidates = [
        book for book in candidates
        if book["id"] != reference_book["id"]
    ]

    if not candidates:
        return []

    index, _ = build_book_index(candidates)

    reference_text = book_to_text(reference_book)

    return search_similar_books(
        reference_text,
        candidates,
        index,
        top_k=top_k
    )

In [ ]:
#Similar-but-different
def remove_reference_author(results, reference_book):
    reference_author = reference_book.get("author", "").lower()

    filtered = []

    for book in results:
        author = book.get("author", "").lower()

        if reference_author and reference_author in author:
            continue

        filtered.append(book)

    return filtered

In [ ]:
def remove_duplicate_titles(results):
    seen = set()
    filtered = []

    for book in results:
        title = book.get("title", "").strip().lower()

        if title in seen:
            continue

        seen.add(title)
        filtered.append(book)

    return filtered

In [ ]:
#filtering
def apply_filters(
    books,
    genre=None,
    language=None,
    max_pages=None,
    min_pages=None
):
    filtered = []

    for book in books:

        if genre:
            subjects = " ".join(
                book.get("subjects", [])
            ).lower()

            if genre.lower() not in subjects:
                continue

        if language:
            languages = [
                str(lang).lower()
                for lang in book.get("languages", [])
            ]

            if language.lower() not in languages:
                continue

        pages = book.get("pages")

        if max_pages is not None:
            if pages is not None and pages > max_pages:
                continue

        if min_pages is not None:
            if pages is not None and pages < min_pages:
                continue

        filtered.append(book)

    return filtered

In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("LLM loaded successfully.")

In [ ]:
def generate_text(messages, max_new_tokens=256):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
QUERY_PARSER_PROMPT = """
You are the query understanding component of a book recommendation system.

Convert the user's request into ONLY valid JSON.

Use exactly these keys:

{
  "reference_title": null,
  "genres": [],
  "mood": [],
  "language": null,
  "max_pages": null,
  "min_pages": null,
  "similar_but_different": false,
  "keywords": []
}

Rules:

- reference_title: a book title only if the user mentions a book they are reading
  or explicitly wants recommendations similar to.
- genres: genres explicitly requested or strongly implied.
- mood: emotional or atmospheric preferences.
- language: requested book language if specified.
- max_pages: maximum number of pages if specified.
- min_pages: minimum number of pages if specified.
- similar_but_different: true when the user asks for something similar
  but different, less fantasy, different author, etc.
- keywords: useful concepts for book retrieval.
- Do not invent preferences.
- Use null when information is unavailable.
- Return JSON only.
"""

In [ ]:
def parse_json_response(text):
    text = text.strip()

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL
    )

    if not match:
        return None

    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

In [ ]:
def parse_user_query(user_query):
    messages = [
        {
            "role": "system",
            "content": QUERY_PARSER_PROMPT
        },
        {
            "role": "user",
            "content": user_query
        }
    ]

    response = generate_text(
        messages,
        max_new_tokens=256
    )

    parsed = parse_json_response(response)

    if parsed is None:
        return {
            "reference_title": None,
            "genres": [],
            "mood": [],
            "language": None,
            "max_pages": None,
            "min_pages": None,
            "similar_but_different": False,
            "keywords": [user_query]
        }

    return parsed

In [ ]:
#test
test_queries = [
    "I want something emotional and historical, preferably Arabic and under 300 pages.",
    "I'm reading The Alchemist and want something similar but with a different author.",
    "I want a dark mystery about a detective."
]

for query in test_queries:
    print("\nUSER:")
    print(query)

    print("\nPARSED:")
    print(
        json.dumps(
            parse_user_query(query),
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:
def build_search_query(parsed, reference_book=None):
    parts = []

    if reference_book:
        parts.append(reference_book.get("title", ""))

        author = reference_book.get("author", "")
        if author:
            parts.append(author)

        subjects = reference_book.get("subjects", [])
        parts.extend(subjects[:8])

    parts.extend(parsed.get("genres", []))
    parts.extend(parsed.get("mood", []))
    parts.extend(parsed.get("keywords", []))

    parts = [
        str(part).strip()
        for part in parts
        if part and str(part).strip()
    ]

    if not parts:
        return "fiction"

    return " ".join(parts[:20])

In [ ]:
def retrieve_candidates(parsed, reference_book=None, candidate_limit=50):
    search_query = build_search_query(
        parsed,
        reference_book
    )

    raw_results = search_books(
        search_query,
        limit=candidate_limit
    )

    candidates = [
        normalize_book(book)
        for book in raw_results.get("docs", [])
    ]

    return candidates

In [ ]:
def get_reference_book(parsed):
    title = parsed.get("reference_title")

    if not title:
        return None

    return find_reference_book(title)

In [ ]:
def build_semantic_query(user_query, parsed, reference_book=None):
    parts = [user_query]

    if parsed.get("genres"):
        parts.append(
            "Genres: " + ", ".join(parsed["genres"])
        )

    if parsed.get("mood"):
        parts.append(
            "Mood: " + ", ".join(parsed["mood"])
        )

    if parsed.get("keywords"):
        parts.append(
            "Keywords: " + ", ".join(parsed["keywords"])
        )

    if reference_book:
        parts.append(
            "Reference book: " + book_to_text(reference_book)
        )

    return " ".join(parts)

In [ ]:
def recommend_books(user_query, top_k=5):
    parsed = parse_user_query(user_query)

    reference_book = get_reference_book(parsed)

    candidates = retrieve_candidates(
        parsed,
        reference_book=reference_book,
        candidate_limit=100
    )

    # Remove the exact reference book
    if reference_book:
        candidates = [
            book for book in candidates
            if book.get("id") != reference_book.get("id")
        ]


    # Remove duplicate titles
    candidates = remove_duplicate_titles(candidates)

    # If the user explicitly asked for a different author,
    # remove the reference author's books.
    if (
        reference_book
        and parsed.get("similar_but_different")
    ):
        candidates = remove_reference_author(
            candidates,
            reference_book
        )

    # If we still have no candidates, return safely.
    if not candidates:
        return {
            "parsed_query": parsed,
            "reference_book": reference_book,
            "recommendations": []
        }

    # Create FAISS index
    index, _ = build_book_index(candidates)

    # Build semantic representation of the user's request
    semantic_query = build_semantic_query(
        user_query,
        parsed,
        reference_book
    )

    # Semantic ranking
    ranked = search_similar_books(
        semantic_query,
        candidates,
        index,
        top_k=min(top_k, len(candidates))
    )

    # Apply explicit filters AFTER semantic ranking.
    ranked = apply_filters(
        ranked,
        genre=None,
        language=parsed.get("language"),
        max_pages=parsed.get("max_pages"),
        min_pages=parsed.get("min_pages")
    )

    return {
        "parsed_query": parsed,
        "reference_book": reference_book,
        "recommendations": ranked[:top_k]
    }

In [ ]:
#test
query = "I'm reading The Alchemist and want something similar but with a different author."

parsed = parse_user_query(query)

print("PARSED QUERY")
print(
    json.dumps(
        parsed,
        indent=2,
        ensure_ascii=False
    )
)

reference_book = get_reference_book(parsed)

print("\nREFERENCE BOOK")

if reference_book:
    print(
        reference_book["title"],
        "-",
        reference_book["author"]
    )
else:
    print("None")

candidates = retrieve_candidates(
    parsed,
    reference_book=reference_book,
    candidate_limit=100
)

print("\nCANDIDATES FROM OPEN LIBRARY:")
print(len(candidates))

for book in candidates[:10]:
    print(
        "-",
        book["title"],
        "|",
        book["author"]
    )

In [ ]:
#test
result = recommend_books(
    "I want a dark mystery about a detective with lots of suspense.",
    top_k=5
)

print(
    json.dumps(
        result["parsed_query"],
        indent=2,
        ensure_ascii=False
    )
)

print("\nRECOMMENDATIONS:\n")

for i, book in enumerate(result["recommendations"], 1):
    print(
        f"{i}. {book['title']} — {book['author']} "
        f"| similarity={book['similarity']:.3f}"
    )

In [ ]:
#test
result = recommend_books(
    "I'm reading The Alchemist and want something similar but with a different author.",
    top_k=5
)

print("Parsed query:")
print(
    json.dumps(
        result["parsed_query"],
        indent=2,
        ensure_ascii=False
    )
)

print("\nReference book:")

if result["reference_book"]:
    print(
        result["reference_book"]["title"],
        "-",
        result["reference_book"]["author"]
    )
else:
    print("No reference book detected.")

print("\nNumber of recommendations:")
print(len(result["recommendations"]))

print("\nRECOMMENDATIONS:\n")

for i, book in enumerate(result["recommendations"], 1):
    print(
        f"{i}. {book['title']} — {book['author']} "
        f"| similarity={book['similarity']:.3f}"
    )

In [ ]:
def generate_explanation(user_query, book, reference_book=None):
    reference_text = ""

    if reference_book:
        reference_text = (
            f"\nReference book:\n"
            f"{book_to_text(reference_book)}"
        )

    prompt = f"""
You are the explanation component of BookMatch.

Explain briefly why the following book is relevant to the user's request.

User request:
{user_query}

Recommended book:
{book_to_text(book)}
{reference_text}

Rules:
- Be concise.
- Mention only information supported by the provided book metadata.
- Do not invent plot details.
- Do not claim that the user has read or liked the book.
- Explain the match in 2 or 3 sentences.
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    return generate_text(
        messages,
        max_new_tokens=120
    )

In [ ]:
#generate explanation for recommendtions
def add_explanations(user_query, result):
    reference_book = result.get("reference_book")

    for book in result["recommendations"]:
        book["explanation"] = generate_explanation(
            user_query,
            book,
            reference_book
        )

    return result

In [ ]:
#test
result = recommend_books(
    "I'm reading The Alchemist and want something similar but with a different author.",
    top_k=5
)

result = add_explanations(
    "I'm reading The Alchemist and want something similar but with a different author.",
    result
)

for i, book in enumerate(result["recommendations"], 1):

    print(f"\n{i}. {book['title']}")
    print(f"Author: {book['author']}")
    print(f"Similarity: {book['similarity']:.3f}")
    print(f"Explanation: {book['explanation']}")

In [ ]:
query = (
    "I'm reading The Alchemist and want "
    "something similar but with a different author."
)

result = recommend_books(
    query,
    top_k=5
)

print("Recommendations found:", len(result["recommendations"]))

if not result["recommendations"]:
    print("\nNo recommendations were found.")
    print("Parsed query:")
    print(
        json.dumps(
            result["parsed_query"],
            indent=2,
            ensure_ascii=False
        )
    )

else:
    result = add_explanations(
        query,
        result
    )

    for i, book in enumerate(
        result["recommendations"],
        1
    ):
        print(f"\n{i}. {book['title']}")
        print(f"Author: {book['author']}")
        print(
            f"Similarity: "
            f"{book['similarity']:.3f}"
        )
        print(
            f"Explanation: "
            f"{book['explanation']}"
        )

In [ ]:
def format_recommendation(book):
    return {
        "title": book.get("title"),
        "author": book.get("author"),
        "year": book.get("year"),
        "pages": book.get("pages"),
        "similarity": round(
            book.get("similarity", 0),
            3
        ),
        "cover_url": get_cover_url(book),
        "book_url": get_book_url(book),
        "explanation": book.get("explanation", "")
    }

In [ ]:
def bookmatch(user_query, top_k=5):
    if not user_query or not user_query.strip():
        return {
            "success": False,
            "message": "Please enter a book request.",
            "recommendations": []
        }

    try:
        result = recommend_books(
            user_query,
            top_k=top_k
        )

        result = add_explanations(
            user_query,
            result
        )

        recommendations = [
            format_recommendation(book)
            for book in result["recommendations"]
        ]

        return {
            "success": True,
            "query": user_query,
            "parsed_query": result["parsed_query"],
            "reference_book": (
                format_recommendation(
                    result["reference_book"]
                )
                if result["reference_book"]
                else None
            ),
            "recommendations": recommendations
        }

    except requests.RequestException:
        return {
            "success": False,
            "message": "Unable to reach the book catalogue.",
            "recommendations": []
        }

    except Exception as e:
        return {
            "success": False,
            "message": f"Recommendation error: {str(e)}",
            "recommendations": []
        }

In [ ]:
#test
response = bookmatch(
    "I want an emotional historical novel, preferably Arabic, under 300 pages.",
    top_k=5
)

print(
    json.dumps(
        response,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
#test
response = bookmatch(
    "I'm reading The Alchemist and want something similar but different.",
    top_k=5
)

print(
    json.dumps(
        response,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
#test
response = bookmatch(
    "Recommend a dark mystery with a detective and suspense.",
    top_k=5
)

for book in response["recommendations"]:
    print("\n" + book["title"])
    print(book["author"])
    print(book["explanation"])

In [ ]:
app = FastAPI(
    title="BookMatch API",
    description="AI-powered book recommendation backend",
    version="1.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

In [ ]:
class RecommendationRequest(BaseModel):
    query: str
    top_k: int = 5

In [ ]:
#health endpoint
@app.get("/")
def root():
    return {
        "message": "BookMatch API is running."
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }

In [ ]:
#recommendation endpoint
@app.post("/recommend")
def recommend(request: RecommendationRequest):
    return bookmatch(
        request.query,
        request.top_k
    )

In [ ]:
def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

time.sleep(3)

print("BookMatch API started on port 8000.")

In [ ]:
#test
response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.json())

In [ ]:
#test
response = requests.post(
    "http://127.0.0.1:8000/recommend",
    json={
        "query": "I want an emotional book about friendship and personal growth.",
        "top_k": 5
    }
)

print(response.status_code)

print(
    json.dumps(
        response.json(),
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

user_secrets = UserSecretsClient()

ngrok_token = user_secrets.get_secret(
    "NGROK_AUTH_TOKEN"
)

ngrok.set_auth_token(ngrok_token)

tunnel = ngrok.connect(8000)

print("Ngrok tunnel:")
print(tunnel)

In [ ]:
public_base_url = tunnel.public_url

print("Public API URL:")
print(public_base_url)

In [ ]:
public_base_url = str(tunnel).split('"')[1]

print("Public API URL:")
print(public_base_url)

In [ ]:
response = requests.get(
    f"{public_base_url}/health"
)

print(response.status_code)
print(response.json())

In [ ]:
response = requests.post(
    f"{public_base_url}/recommend",
    json={
        "query": "I want a dark mystery with suspense.",
        "top_k": 5
    }
)

print(response.status_code)

data = response.json()

for book in data["recommendations"]:
    print(
        book["title"],
        "—",
        book["author"]
    )